# Recurrent Neural Networkによる電力予測

---
## 目的
Recurrent Neural Networkを使って電力予測を行う．ここで，今回は最も基本的な再帰型ニューラルネットワークである`RNNCell`を使用する．
また，PyTorchで使用されるデータセットオブジェクトの作成を行う．

## モジュールのインポート
はじめに必要なモジュールをインポートしたのち，GPUを使用した計算が可能かどうかを確認する．

In [ ]:
from time import time
from os import path
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import gdown

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

### データのダウンロードと確認

プログラムの動作に必要なデータをダウンロードし，zipファイルを解凍する．

In [ ]:
if not path.isdir('./BEMS_data'):
    gdown.download('https://drive.google.com/uc?id=1oMM1Xu2-hIe4Of2mfznvBNGCQIe54O1f', 'BEMS_data.zip', quiet=False)
    with zipfile.ZipFile('BEMS_data.zip') as f:
        f.extractall('./')

データを確認してみます．最初の７つの値が曜日のone-hot vector，次の２４個の値は時間（Hour）のone-hot vector，残りが電力，気温，湿度です．

In [ ]:
tmp_data = np.load("./BEMS_data/BEMS_RNN_train_data.npy")      # 学習用の入力データを読み込み
tmp_label = np.load("./BEMS_data/BEMS_RNN_train_labels.npy")   # 学習用のラベルを読み込み
print(tmp_data[0:5,:])     # データの一番目（1~5時刻目）の中身の表示
print(tmp_data.shape)  # データ（全部）の配列サイズの表示
print(tmp_label[0:5])    # ラベルの一番目（1~5時刻目）の中身の表示
print(tmp_label.shape) # ラベル（全部）の配列サイズの表示

## データセットオブジェクトの作成

電力データセットに対する，PyTorchのデータセットオブジェクト (`torch.utils.data.Dataset`) を作成します．
`Dataset`は，指定したデータセットを読み込み，学習やテストのためにデータを準備し生成するためのクラスです．
これまでの実習で使用したMNISTやCIFARデータセットはPyTorch (torchvision) 内に準備されているデータセットオブジェクトでした．
今回用いるデータセットは，torchvisonには存在しないため，自身で定義を行います．

**initの定義**

まず，`__init__`関数により，必要なデータを読み込みます．
この時，`__init__`関数の引数を指定します．引数は以下の通りです．

* root：データのあるフォルダを指定
* train：学習用データを使用するかどうか
* delay：何時刻先のラベルを正解とするか
* time_window：1サンプルあたり何時刻のデータを準備するか

まず，`root`および`train`変数から，学習またはテストデータを読み込みます．
その後，`delay`で指定した時刻を元に正解データを準備します．
最後に，`time_window`で指定した時間窓で1サンプルとなるように，データを作成し，`self.data`および`self.label`にデータを格納します．
これにより，`self.data`，`self.label`に入力データおよび正解データを格納します．

**getitemの定義**

`__getitem__`関数で，指定したインデックス（`item`）のデータを取り出し，返します．

**lenの定義**

`__len__`関数は，このデータセットが保有するサンプル数を返すように定義を行います．


In [ ]:
class BEMSDataset(torch.utils.data.Dataset):

    def __init__(self, root="./data", train=True, delay=1, time_window=10):
        super().__init__()
        # 引数で与えた情報をデータセット内で保持するよう，クラス変数に保存
        self.root = root
        self.train = train
        self.delay = delay
        self.time_window = time_window

        # データの読み込み
        if self.train:  # 学習用データ (train=True) の場合
            data_src = np.load(path.join(self.root, 'BEMS_RNN_train_data.npy'))
            label_src = np.load(path.join(self.root, 'BEMS_RNN_train_labels.npy'))
        else:           # テスト用データ (train=False) の場合
            data_src  = np.load(path.join(self.root, 'BEMS_RNN_test_data.npy'))
            label_src = np.load(path.join(self.root, 'BEMS_RNN_test_labels.npy'))

        # self.delay分だけデータとその正解ラベルの時刻をずらして準備する
        data_src = np.asarray(data_src[:-self.delay])   # 0 ~ 後ろからself.deley番目まで
        label_src = np.asarray(label_src[self.delay:])  # self.delay ~ 最後まで

        # self.time_windowの長さで一つのサンプルになるようにデータを区切ってひとつづつ準備
        self.data = []
        self.label = []
        for frame_i in range(len(data_src) - self.time_window):
            self.data.append(data_src[frame_i:frame_i+self.time_window])
            self.label.append(label_src[frame_i:frame_i+self.time_window])

        # リストに格納されたデータをnumpy配列形式に変換
        self.data = np.asarray(self.data)
        self.label = np.asarray(self.label)

    def __getitem__(self, item):
        # item番目のデータとラベルを取得
        d = self.data[item, :]
        l = self.label[item, :]
        return d, l

    def __len__(self):
        return self.data.shape[0]  # self.dataの配列の1次元目のサイズ（サンプル数）を返す

### データセットの読み込み
学習データ（BEMSDataset）を読み込みます．

In [ ]:
time_window = 10

train_data = BEMSDataset(root="./BEMS_data", train=True, delay=1, time_window=time_window)
test_data = BEMSDataset(root="./BEMS_data", train=False, delay=1, time_window=1)

## ネットワークモデルの定義

再帰型ニューラルネットワークを定義します．
ここでは，`RNNCell`の層1層，全結合層1層から構成されるネットワークとします．

PyTorchには，時系列データ全体を一度に入力できる`nn.RNN`（内部でtime_window分のループを自動的に処理する）と，1時刻分の入力のみを受け取る`nn.RNNCell`（時刻ごとのループを自分で記述する必要がある）の2種類が用意されています．ここでは，RNNが時刻ごとに隠れ状態を更新していく様子を明示的に確認できるように，`nn.RNNCell`を使用します．

**initの定義**

再帰型NN層 (`self.rnn`) として`nn.RNNCell`を定義します．

**forwardの定義**

`forward`関数では，定義した層を接続して処理するように記述します．

このとき，全結合層から出力された結果にくわえて，`self.rnn`の隠れ状態も同時に返し，次時刻への入力へと使用します．

In [ ]:
class RNN(nn.Module):
    def __init__(self, n_hidden):
        super(RNN, self).__init__()

        self.rnn = nn.RNNCell(34, n_hidden)
        self.l1 = nn.Linear(n_hidden, 1)

    def forward(self, x, hx):
        hx = self.rnn(x, hx)
        h = self.l1(hx)
        return h, hx

## ネットワークの作成
上のプログラムで定義したネットワークを作成します．

In [ ]:
n_hidden = 128

model = RNN(n_hidden).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

## 学習
先ほど定義したデータセットと作成したネットワークを用いて，学習を行います．

1回の誤差を算出するデータ数（ミニバッチサイズ）を20，学習エポック数を20とします．
また，1サンプルあたりのデータの長さ（time window）は上で指定したように10に指定します．

次にデータローダーを定義します．
データローダーでは，上で読み込んだデータセット（`train_data`）を用いて，for文で指定したミニバッチサイズでデータを読み込むオブジェクトを作成します．
この時，`shuffle=True`と設定することで，読み込むデータを毎回ランダムに指定します．

次に，誤差関数を設定します．
今回は，連続値を出力する回帰問題をあつかうため，`MSELoss`を`criterion`として定義します．

各更新において，学習用データと教師データをそれぞれ`data`と`label`とします．
まず，RNNの隠れ状態である`hx`を`torch.zeros`を用いて初期化します．
この時，1次元目のサイズはバッチサイズに対応するように，`data`のサイズから自動的に決定します．

その後，学習モデルに`data`を与えて予測値yを取得します．
今回はRNNを用いて時系列データを順次処理するため，for文を用いて，各時刻のデータを順番に入力し，結果を得ます．
そして，各時刻の予測値yと教師ラベルとの誤差を`criterion`で算出し，time_window分の誤差を`loss`に累積します．
そして，誤差をbackward関数で逆伝播し，ネットワークの更新を行います．

In [ ]:
# ミニバッチサイズ・エポック数の設定
batch_size = 20
epoch_num = 20

# データローダーの設定
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)

# 誤差関数の設定
criterion = nn.MSELoss().to(device)

# ネットワークを学習モードへ変更
model.train()

start = time()
for epoch in range(1, epoch_num + 1):
    sum_loss = 0.0

    for data, label in train_loader:
        data = data.to(device)
        label = label.to(device)
        hx = torch.zeros(data.size(0), n_hidden, device=device)

        loss = 0
        for idx_window in range(time_window):
            y, hx = model(data[:, idx_window, :], hx)
            loss += criterion(y, label[:, idx_window:idx_window+1])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()

    print('epoch: {}, mean loss: {:.4f}, elapsed_time: {:.4f}'.format(epoch, sum_loss / len(train_loader), time() - start))

## テスト
学習したネットワークモデルを用いて評価（予測結果の可視化）を行います．可視化にはmatplotlibを用います．

テスト用データセット（`test_data`）は`time_window=1`で作成しているため，1サンプルが1時刻分のデータになっています．以下では，「1ステップ先予測」と「自己回帰的な複数ステップ先予測」の2つの方法で評価を行います．

### 1ステップ先予測
毎時刻，真のデータ（曜日・時刻・電力・気温・湿度）をそのままモデルに入力して1時刻先を予測します．これは，直前の状態が正確にわかっている前提での予測（教師強制; teacher forcing）であり，モデル自体の性能を確認するには有用ですが，遠い未来の電力を予測したいという実際の運用場面を反映したものではありません．

In [ ]:
# データローダーの設定
test_loader = torch.utils.data.DataLoader(test_data, batch_size=1, shuffle=False)

# ネットワークを評価モードへ変更
model.eval()

pred_duration = 500  # 予測を行う期間（10000時刻分のデータ全ては使わない）

prediction_result = []  # 1ステップ先予測（毎時刻，真のデータを入力）

hx = torch.zeros(1, n_hidden, device=device)

with torch.no_grad():
    for t, (data, label) in enumerate(test_loader):
        data = data.to(device)
        y, hx = model(data[:, 0, :], hx)
        prediction_result.append(y.item())

        if (t+1) % pred_duration == 0:
            break

prediction_result = np.array(prediction_result).flatten()

# 結果の表示
plt.figure()
plt.title('RNN (1-step)')
plt.plot(prediction_result.tolist(), color='blue', label='pred (1-step)')
plt.plot(test_data.label[:pred_duration], color='red', label='true')
plt.legend()
plt.show()

### 自己回帰的な複数ステップ先予測
より現実的な評価として，最初の`warmup_steps`ステップだけ真のデータをそのまま入力して`hx`を準備し，その後の`pred_duration`ステップは真の電力値を使わず，直前の自分自身の予測値を次の入力の電力の要素として使い回しながら予測を続けます．曜日・時刻は未来でも既知の暦情報のためそのまま真の値を使用し，気温・湿度はこのモデルが予測する対象ではないため真の値を使用します．この自己回帰予測により，モデルが自身の予測誤差を蓄積させながら，遠い未来の電力をどれだけ正確に予測できるかを確認できます．

In [ ]:
power_idx = 31        # 入力ベクトル中の電力の要素番号（曜日one-hot 7 + 時刻one-hot 24 の次）
warmup_steps = 250     # 最初に真のデータを入力してhxを準備するステップ数
pred_duration = 250   # その後，予測値を自己回帰的に入力し続けるステップ数

prediction_result_ar = []  # 自己回帰的な複数ステップ先予測

hx_ar = torch.zeros(1, n_hidden, device=device)
prev_pred = None  # 自己回帰予測中に直前の予測値を保持しておくための変数

with torch.no_grad():
    for t, (data, label) in enumerate(test_loader):
        data = data.to(device)

        if t < warmup_steps:
            # 最初のwarmup_stepsステップは真のデータをそのまま入力し，hxを準備する
            input_ar = data[:, 0, :]
        else:
            # 以降は電力の要素を直前の自分自身の予測値に置き換えて入力する
            input_ar = data[:, 0, :].clone()
            input_ar[:, power_idx] = prev_pred

        y_ar, hx_ar = model(input_ar, hx_ar)
        prev_pred = y_ar.item()
        prediction_result_ar.append(y_ar.item())

        if t + 1 == warmup_steps + pred_duration:
            break

prediction_result_ar = np.array(prediction_result_ar).flatten()

# 結果の表示
plt.figure()
plt.title('RNN (autoregressive, warmup={}, pred_duration={})'.format(warmup_steps, pred_duration))
plt.plot(prediction_result_ar.tolist(), color='green', label='pred (autoregressive)')
plt.plot(test_data.label[:warmup_steps + pred_duration], color='red', label='true')
plt.axvline(warmup_steps, color='gray', linestyle='--', label='warmup end')
plt.legend()
plt.show()

## 課題

1. `nn.RNNCell`を`nn.LSTMCell`や`nn.GRUCell`に変更して，結果を確認しましょう．
    * `LSTMCell`はセル状態`cx`も入出力する点に注意してください．
2. 電力予測について，入力データを現在の電力・気温・湿度のみ入力してみましょう．
3. `warmup_steps`や`pred_duration`の値を変えて，自己回帰的な複数ステップ先予測の誤差がどのように変化するか確認してみましょう．